In [0]:
%pip install langchain
%pip install databricks-vectorsearch
dbutils.library.restartPython()

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
import os
from dotenv import load_dotenv
from langchain_openai import AzureChatOpenAI
from langchain.tools import tool
from langgraph.checkpoint.memory import MemorySaver
from langgraph.prebuilt import create_react_agent
from databricks.vector_search.client import VectorSearchClient

load_dotenv()

ROLL_NUMBER      = "27100030"
CATALOG_NAME_RAW = f"{ROLL_NUMBER}-pa3"
endpoint_name    = f"{ROLL_NUMBER}_vector_endpoint"
index_name       = f"{CATALOG_NAME_RAW}.vector_index.fixed_vector_index"
TEXT_VOLUME      = f"/Volumes/{CATALOG_NAME_RAW}/default/text_documents"


In [0]:
def retrieve_formatted_context(question: str, topk: int = 3):
    results = VectorSearchClient(disable_notice=True).get_index(
        endpoint_name=endpoint_name,
        index_name=index_name,
    ).similarity_search(
        query_text=question,
        columns=["ChunkText"],
        num_results=topk,
    )
    docs = results.get("result", {}).get("data_array", []) or []
    raw_docs = [doc[1] for doc in docs if len(doc) > 1]
    return "\n\n---\n\n".join(raw_docs)


@tool
def vector_search_tool(query: str) -> str:
    """Search the course vector store for passages relevant to the query and return them as a formatted string."""
    try:
        context = retrieve_formatted_context(query)
        return context if context else "No matching documents were found."
    except Exception as e:
        return f"Vector search failed: {e}"

@tool
def read_fallback_document(category: str) -> str:
    """Read the Azure or Databricks documentation file for questions the vector store cannot answer. Pass 'azure' or 'databricks'."""
    file_map = {
        "databricks": f"{TEXT_VOLUME}/databricks_info.txt",
        "azure":      f"{TEXT_VOLUME}/azure_info.txt"
    }
    
    category = (category or "").strip().lower()
    file_path = file_map.get(category)
    
    if not file_path:
        return f"No documentation found for category: {category}"
        
    try:
        with open(file_path, "r") as f:
            return f.read()
    except Exception as e:
        return f"Error reading document: {e}"
    

def build_stateful_mcp_agent():
    llm = AzureChatOpenAI(
        azure_endpoint=os.environ.get("AZURE_OPENAI_ENDPOINT"),
        azure_deployment=os.environ.get("AZURE_OPENAI_CHAT_DEPLOYMENT"), 
        api_version=os.environ.get("OPENAI_API_VERSION"), 
        temperature=0.0
    )
    
    tools = [vector_search_tool, read_fallback_document]
    memory = MemorySaver()
    
    mcp_agent = create_react_agent(llm, tools, checkpointer=memory)
    return mcp_agent

agent = build_stateful_mcp_agent()


/home/spark-6ca3a15b-fb8d-4493-9264-14/.ipykernel/3942/command-7873123781764793-2391943109:56: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  mcp_agent = create_react_agent(llm, tools, checkpointer=memory)


In [0]:
# Summary: Executes queries against the agent while maintaining a consistent session thread.
def execute_mcp_queries(mcp_agent, queries: list, thread_id: str):
    config = {"configurable": {"thread_id": thread_id}}
    
    for query in queries:
        print(f"\nUser: {query}")
        payload = {"messages": [{"role": "user", "content": query}]}
        
        response = mcp_agent.invoke(payload, config=config)
        final_message = response["messages"][-1].content
        
        print(f"Agent: {final_message}")


In [0]:
validation_queries = [
    "What tools do you have?",
    "What is searchAgent-X?",
    "What does this agent do specifically?",
    "Use your tool to search for information about Azure.",
    "Based on those search results, summarize its main features.",
    "What is Databricks?",
    "What core features differentiate both products?"
]

print("--- TESTING STATEFUL MCP AGENT ---")
execute_mcp_queries(agent, validation_queries, "mcp_test_session_01")

--- TESTING STATEFUL MCP AGENT ---

User: What tools do you have?
Agent: I have access to the following tools:

1. Vector Search Tool: I can search a course vector store for passages relevant to your query and return them as a formatted string.

2. Read Fallback Document: If the vector store cannot answer your question, I can read the Azure or Databricks documentation files to find the information you need.

3. Multi-Tool Use: I can run multiple tools simultaneously if needed, especially if they can operate in parallel.

If you have any specific questions or need information, just let me know!

User: What is searchAgent-X?
Agent: I currently do not have information about "searchAgent-X" in the available resources. It might be a specialized or proprietary tool not covered in the course materials or documentation I can access.

If you can provide more context or details about where you encountered "searchAgent-X" or its application area, I can try to assist you further.

User: What does 

### Bonus Task 3: Agent Validation

The output above shows the agent:
- Calling `vector_search_tool` for domain questions (SearchAgent-X, Azure, Databricks)
- Using `read_fallback_document` when the vector store lacks coverage
- Retaining context across turns — "Based on those search results, summarize its main features" 
  correctly refers back to the Azure search from the previous query without re-searching
- Correctly listing its own tools when asked "What tools do you have?"

ok
